# Multi-seed aggregation and cross-architecture statistics

Cross-architecture statistical comparison for the three model families benchmarked in Chapter 3 of the thesis:

* **qcnn** — Hybrid Q-CNN trained by `HQCNN.ipynb`
* **ccnn** — Classical-CNN ablation trained by `qiskit_one_q.ipynb`
* **pure_q** — Pure-quantum reference (Sebastianelli et al.) trained by `esa_modello.ipynb`

Each of the three notebooks deposits one CSV per seed under `multirun_csv/<arch>/`. This notebook loads those CSVs and produces (i) the across-run summary tables, (ii) the Wilcoxon signed-rank paired tests across architectures, and (iii) the combined `cross_architecture_with_std.png` plot for the manuscript.

## Statistical protocol

The replication unit is the seed: for each architecture we have $R=10$ independent runs, indexed by seeds $\{42, 43, \ldots, 51\}$. The train/validation split is fixed across seeds, so any across-run dispersion comes from weight initialisation and mini-batch ordering only. We report:

* **Mean and across-run standard deviation** of the final-epoch validation accuracy, per architecture (the figure that appears as `mean ± sigma` in the thesis).
* **Bootstrap 95% percentile CI** on the across-run mean (descriptive, based on $10^4$ resamples).
* **Wilson 95% CI** on the single-run accuracy, taken as the median across seeds (descriptive single-run uncertainty figure; with $N_{\text{val}}\sim 20$ the half-width is large by construction).
* **Wilcoxon signed-rank paired test** on the $R=10$ across-seed accuracy differences between pairs of architectures. The pairing is on the seed (same initialisation, same shuffle) so that the test cancels seed-induced variation. With $R=10$ the test uses the exact distribution and is the scientifically appropriate inferential procedure in this setup.

We **do not** report McNemar's exact test on per-item discordant counts: with $N_{\text{val}}\sim 20$ and accuracy $\sim 0.9$ the discordant counts are 2–3 in total and McNemar has essentially zero statistical power.

In [ ]:
import os, sys
import numpy as np

_ROOT = os.path.abspath(os.path.join(os.getcwd(), '.'))
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

import multirun

CSV_DIR = 'multirun_csv'

agg = {}
for arch in ('qcnn', 'ccnn', 'pure_q'):
    try:
        agg[arch] = multirun.load_aggregated(CSV_DIR, arch)
        print(f'[OK] {arch:>6s}: R={agg[arch].n_runs}, epochs={agg[arch].n_epochs}, '
              f'n_val={agg[arch].n_val}, '
              f'final mean={agg[arch].final_val_acc.mean():.4f}')
    except FileNotFoundError as e:
        print(f'[MISS] {arch}: {e}')

if len(agg) < 2:
    raise RuntimeError('Need at least two architectures to compare; '
                       'run the three single-arch notebooks first.')

In [ ]:
# Per-architecture summary table
print(multirun.summary_table(agg))

In [ ]:
# Wilcoxon signed-rank paired tests across architectures.
# Pairing is on the seed (same i across architectures).
import itertools

pairs = [
    ('qcnn', 'ccnn'),
    ('qcnn', 'pure_q'),
    ('ccnn', 'pure_q'),
]

print(f'{"comparison (A vs B)":>22s}  {"R":>3s}  '
      f'{"mean(A)":>8s}  {"mean(B)":>8s}  '
      f'{"mean diff":>10s}  {"W stat":>8s}  {"p (two-sided)":>14s}')
print('-' * 90)
for (A, B) in pairs:
    if A not in agg or B not in agg:
        continue
    res = multirun.wilcoxon_paired(agg[A].final_val_acc,
                                   agg[B].final_val_acc)
    mean_diff = float(np.mean(agg[A].final_val_acc - agg[B].final_val_acc))
    print(f'{(A + " vs " + B):>22s}  {res["n_pairs"]:>3d}  '
          f'{agg[A].final_val_acc.mean():>8.4f}  {agg[B].final_val_acc.mean():>8.4f}  '
          f'{mean_diff:>+10.4f}  {res["statistic"]:>8.1f}  {res["p_value"]:>14.4g}')

## Reading the p-values

Each $p$-value above is the two-sided exact $p$-value from the Wilcoxon signed-rank test on the $R=10$ across-seed accuracy differences. A small $p$-value (say $\leq 0.05$) means that the sign of the difference is consistent across seeds; it does **not** quantify the magnitude of the difference, which is reported separately by the across-run mean and bootstrap CI in the previous table.

Important: with $R=10$ the smallest achievable two-sided exact $p$-value is $2/2^{10}\approx 0.002$ (in the case where all 10 paired differences have the same sign). Conversely, a $p$-value $\sim 0.5$ does not establish that the two architectures are equivalent; it only says that the present data do not establish a consistent sign. The manuscript reports these tests as a qualitative robustness check and complements them with the across-run mean and dispersion.

In [ ]:
# Combined plot: three architectures, validation accuracy with band.
FIG_PATH = '../PhDThesis/chapters/qa_figures/cross_architecture_with_std.png'
multirun.plot_three_architectures(agg, output_path=FIG_PATH)
print(f'Figure written to: {FIG_PATH}')

In [ ]:
# LaTeX-friendly snippet of the cross-architecture summary, ready to paste
# into Cap.3 (Sec. Results and Statistical Robustness).

lines = []
lines.append(r'\begin{tabular}{lcccc}')
lines.append(r'\toprule')
lines.append(r'Architecture & $R$ & $N_{\mathrm{val}}$ & mean $\pm$ std & bootstrap 95\% CI \\')
lines.append(r'\midrule')
label_of = {'qcnn': 'Hybrid Q-CNN',
            'ccnn': 'Classical CNN (ablation)',
            'pure_q': 'Pure quantum (Sebastianelli)'}
for arch in ('qcnn', 'ccnn', 'pure_q'):
    if arch not in agg:
        continue
    a = agg[arch]
    boot = multirun.bootstrap_ci_mean(a.final_val_acc)
    lines.append(f'{label_of[arch]} & {a.n_runs} & {a.n_val} & '
                 f'${boot["mean"]:.3f} \\pm {boot["std"]:.3f}$ & '
                 f'$[{boot["ci_low"]:.3f},\\,{boot["ci_high"]:.3f}]$ \\\\')
lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
print('\n'.join(lines))